# Replicate the Transformer budget comparison across seeds

Reuse the completed seed-42 comparison. Train both budgets with seeds 43 and 44.
Keep width 16, context ten, features, learning rate, folds, and checkpoint rules fixed.
New work is twenty fits, with a maximum of 600 epochs. OPAL remains disabled.
These are initialization repeats on the same people, not fifteen independent populations.
Read docs/transformer-budget-seeds.md before enabling preparation or training.


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("transformer_seed_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/transformer-budget-seeds.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/transformer-budget-seeds"  # Select the PR branch before merge
RUN_SETUP = False
RUN_CHECK = True
RUN_PREPARATION = False
RUN_TRAINING = False
logger.info("Repository: %s", ROOT)
logger.info("Run setup: %s", RUN_SETUP)
logger.info("Run preparation: %s", RUN_PREPARATION)
logger.info("Run training: %s", RUN_TRAINING)

## Optional setup

Use the current checkout locally. Select experiment/transformer-budget-seeds while reviewing this PR.
Use main after merge. Existing checkouts are not reset or switched automatically.
On a fresh machine, prepare the source cohort features through notebook 07 first.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")


## Preflight without training

Load development features and run a no-gradient forward pass.
The random model has no reported classification performance. No checkpoint or optimizer is created.


In [ ]:
if RUN_CHECK:
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
        --config configs/transformer-budget-candidate-seed-43.yaml --check
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Transformer preflight failed")

## Inspect the replication plan

Planning reads metadata and configurations only. It does not prepare files or train models.
Every new fold assignment must match seed 42. Only the training seed changes within each budget.


In [ ]:
import pandas as pd

from sleep_classifier.budget_seed_report import BudgetSeedReport
from sleep_classifier.grouped_validation import GroupedValidationExperiment
from sleep_classifier.paired_architecture_report import PairedArchitectureReport

study = BudgetSeedReport(ROOT / "configs/transformer-budget-seeds.yaml")
plan = study.plan()
display(pd.DataFrame(plan["cases"]).set_index("seed"))
logger.info("New fits: %s; maximum new epochs: %s", plan["new_fits"], plan["maximum_new_epochs"])
expected_folds = None
for case in study.cases:
    for role in ("reference", "candidate"):
        grouped_plan = GroupedValidationExperiment(Path(case[f"{role}_config"])).plan()
        if expected_folds is None:
            expected_folds = grouped_plan["folds"]
        if grouped_plan["folds"] != expected_folds:
            raise ValueError("Study folds differ from the measured seed-42 reference")
logger.info("All six grouped configurations use the same participant fold assignments")


## Prepare only the new seed folds

Enable RUN_PREPARATION once for fresh output directories.
Seed 42 is reused and is excluded from preparation and training.


In [ ]:
if RUN_PREPARATION:
    %cd "{ROOT}"
    for case in study.cases:
        if case["reuse"]:
            continue
        for role in ("reference", "candidate"):
            grouped_config = case[f"{role}_config"]
            logger.info("Preparing seed %s, %s", case["seed"], role)
            !uv run --locked --group gru --group notebook python \
                -m sleep_classifier.grouped_validation --config "{grouped_config}" --prepare
            if get_ipython().user_ns["_exit_code"] != 0:
                raise RuntimeError("Seed preparation failed; inspect the output before retrying")
else:
    logger.info("Preparation skipped")


## Train the new paired runs when ready

Enable RUN_TRAINING and execute the settings cell, then execute this cell interactively.
Runs are sequential: reference then candidate for seed 43, followed by seed 44.
The first failed command stops the cell. Existing results are refused; automatic resume is not supported.


In [ ]:
if RUN_TRAINING:
    %cd "{ROOT}"
    for case in study.cases:
        if case["reuse"]:
            continue
        for role in ("reference", "candidate"):
            grouped_config = case[f"{role}_config"]
            logger.info("Training seed %s, %s", case["seed"], role)
            !uv run --locked --group gru --group notebook python -u \
                -m sleep_classifier.grouped_validation --config "{grouped_config}" --train
            if get_ipython().user_ns["_exit_code"] != 0:
                raise RuntimeError("Seed training failed; completed runs are retained")
else:
    logger.info("Training skipped: enable RUN_TRAINING when ready")


## Inspect complete and pending pairs

Each completed pair is independently verified from its saved predictions.
An existing seed-42 result does not mean the new seeds have run.
The full study report requires all three pairs to finish.


In [ ]:
statuses = study.status()
display(pd.DataFrame(statuses).set_index(["seed", "recipe"]))
seed_rows = []
class_rows = []
fold_rows = []
for case in study.cases:
    pair_statuses = [row for row in statuses if row["seed"] == case["seed"]]
    if not all(row["status"] == "complete" for row in pair_statuses):
        continue
    paired = PairedArchitectureReport(
        Path(case["reference_summary"]), Path(case["candidate_summary"]),
        comparison="training_budget",
    ).build()
    reference_mean = pd.DataFrame(paired["folds"])["reference_macro_f1"].mean()
    candidate_mean = pd.DataFrame(paired["folds"])["candidate_macro_f1"].mean()
    seed_rows.append({"seed": case["seed"], "reference_mean_macro_f1": reference_mean,
                      "candidate_mean_macro_f1": candidate_mean,
                      "paired_gain": paired["paired"]["mean_macro_f1_difference"],
                      "participants_improved": sum(row["macro_f1_difference"] > 0
                                                   for row in paired["participants"])})
    for role in ("reference", "candidate"):
        for stage, scores in paired[role]["classes"].items():
            class_rows.append({"seed": case["seed"], "recipe": role,
                               "stage": ["Wake", "Light", "Deep", "REM"][int(stage)], **scores})
    fold_rows.extend({"seed": case["seed"], **row} for row in paired["folds"])
if seed_rows:
    display(pd.DataFrame(seed_rows).set_index("seed"))
    display(pd.DataFrame(class_rows).set_index(["seed", "recipe", "stage"]))
    display(pd.DataFrame(fold_rows).set_index(["seed", "fold"]))


In [ ]:
if all(row["status"] == "complete" for row in statuses):
    result = study.build()
    study.output_dir.mkdir(parents=True, exist_ok=True)
    output = study.output_dir / "summary.json"
    output.write_text(json.dumps(result, indent=2) + "\n")
    display(pd.DataFrame([{
        "mean_seed_paired_gain": result["mean_seed_macro_f1_difference"],
        "sample_std_seed_paired_gain": result["sample_std_seed_macro_f1_difference"],
    }]))
    logger.info("Validated seed comparison saved: %s", output)
else:
    logger.info("Study incomplete: seed-level aggregate requires all three completed pairs")


In [ ]:
if seed_rows:
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    seed_table = pd.DataFrame(seed_rows).set_index("seed")
    seed_table[["reference_mean_macro_f1", "candidate_mean_macro_f1"]].plot.bar(
        ax=axes[0], ylim=(0, 1),
    )
    axes[0].set(xlabel="Training seed", ylabel="Mean development fold macro-F1")
    classes = pd.DataFrame(class_rows)
    rem = classes[classes["stage"] == "REM"].pivot(index="seed", columns="recipe", values="recall")
    rem.plot.bar(ax=axes[1], ylim=(0, 1))
    axes[1].set(xlabel="Training seed", ylabel="Pooled development REM recall")
    figure.tight_layout()
    plt.show()


## Export HTML without training

The export runs the no-gradient preflight and report cells.
It skips setup, preparation, training, and this export command.


In [ ]:
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/15_transformer_budget_seeds.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Seed report export failed")
